# Claude spend ledger

Spend per project from the claude-box ledger: this month and all time.
The ledger is opened read-only. Set `LEDGER_DB` to point at another file, e.g. `data/ledger_example.sqlite`.

In [ ]:
import os
from datetime import date, datetime, timedelta, UTC
from pathlib import Path
from zoneinfo import ZoneInfo

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns
import seaborn.objects as so
from IPython.display import Markdown, display

from ledger_view.ledger import (
    BUDGET_USD, TZ, budget_pace, daily_spend, daily_total, group_small, load_hourly,
    load_sessions, month_days, month_progress, month_start,
    monthly_summary, open_ledger, project_colors, spend_by_model, spend_by_project,
    workday_rolling_mean,
)
from ledger_view.forecast import (
    BayesForecaster, BootstrapForecaster, EtsForecaster, Forecaster, InsufficientHistory,
    LinearForecaster, ModelName, backtest, backtest_origins, backtest_summary, fan,
    future_days, history_from_paced, prob_over,
)

DB = Path(os.environ.get("data/ledger.sqlite", "data/ledger.sqlite"))
NOW = datetime.now(UTC)
BUDGET = BUDGET_USD  # USD per month
SEED = 2026  # fixed, so the forecast bands do not move between runs on the same data
N_PATHS = 2000  # simulated paths per forecast

sns.set_theme(style="whitegrid", rc={"grid.color": "#e6e5e0", "axes.edgecolor": "#c3c2b7"})
_ = pl.Config.set_float_precision(2)

In [ ]:
with open_ledger(DB) as conn:
    hourly = load_hourly(conn, TZ)
    sessions = load_sessions(conn)

since = month_start(NOW, TZ)
all_time = spend_by_project(hourly)
this_month = spend_by_project(hourly, since)
colors = project_colors(all_time["project"].to_list())

display(
    Markdown(
        f"**Total:** \\${all_time['cost_usd'].sum():,.2f}<br>\n"
        f"**{since:%B %Y}:** \\${this_month['cost_usd'].sum():,.2f}<br>\n"
        f"**Sessions:** {sessions.height}<br>\n"
        f"**Data to:** {hourly['ts'].max():%Y-%m-%d %H:%M}"
    )
)

In [ ]:
print(this_month["project"])

In [ ]:
with open_ledger(DB) as conn:
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
    print(cursor.fetchall())
    cursor.close()

## Budget

All projects, backfill included. The monthly budget is spread evenly over the month's working days
(Mon–Fri, minus Norwegian public holidays).

In [ ]:
today = NOW.astimezone(ZoneInfo(TZ)).date()
month = today.replace(day=1)
month_end = month_days(month.year, month.month)[-1]
first = hourly["ts"].min().date().replace(day=1)

paced = budget_pace(daily_total(hourly, first, month_end), BUDGET)
summary = monthly_summary(paced.filter(pl.col("day") <= today), BUDGET)
current = paced.filter(pl.col("month") == month)
to_date = current.filter(pl.col("day") <= today)

spent = to_date["cost_usd"].sum()
pace_budget = to_date["cum_budget"].max()

# Today is not over, so forecast from yesterday's close: today is the first forecast day.
# The headline forecast is the Bayesian model; the bootstrap stands in while history is too short.
cutoff = today - timedelta(days=1)
history = history_from_paced(paced, cutoff)
rest = tuple(d for d in month_days(month.year, month.month) if d >= today)
spent_before = to_date.filter(pl.col("day") < today)["cost_usd"].sum()
try:
    headline_model = "Bayesian"
    paths = BayesForecaster()(history, rest, N_PATHS, np.random.default_rng(SEED))
except InsufficientHistory:
    headline_model = "bootstrap"
    paths = BootstrapForecaster()(history, rest, N_PATHS, np.random.default_rng(SEED))
month_fan = fan(spent_before, paths, rest)
p05, p50, p95 = np.quantile(month_fan.month_end, [0.05, 0.5, 0.95])

display(
    Markdown(
        f"**{month:%B}:** \\${spent:,.2f} of \\${BUDGET:,.0f} ({spent / BUDGET:.0%})<br>\n"
        f"**Budget to date:** \\${pace_budget:,.2f} "
        f"({'over' if spent > pace_budget else 'under'} by \\${abs(spent - pace_budget):,.2f})<br>\n"
        f"**Forecast for {month:%B}** ({headline_model}): median \\${p50:,.0f} "
        f"(90%: \\${p05:,.0f}–\\${p95:,.0f}), "
        f"chance of going over budget {prob_over(month_fan.month_end, BUDGET):.0%}"
    )
)

In [ ]:
def draw_fan(ax: plt.Axes, f, start: date, start_spend: float, color: str) -> None:
    """90% and 50% bands and the median of cumulative spend, from today's spend onward."""
    if not f.days:
        return
    xs = [start, *f.days]
    q = {k: [start_spend, *v] for k, v in f.quantiles.items()}
    ax.fill_between(xs, q[0.05], q[0.95], color=color, alpha=0.15, lw=0, label="90% band")
    ax.fill_between(xs, q[0.25], q[0.75], color=color, alpha=0.30, lw=0, label="50% band")
    ax.plot(xs, q[0.5], color=color, lw=2, ls=":", label="Median forecast")


fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(current["day"], current["cum_budget"], color="#8a8983", lw=2, ls="--", label="Budget pace")
ax.axhline(BUDGET, color="#8a8983", lw=1)
ax.plot(to_date["day"], to_date["cum_spend"], color="#2a78d6", lw=2, label="Spent")
draw_fan(ax, month_fan, cutoff, spent_before, "#2a78d6")

end_labels = [
    (today, spent, f"\\${spent:,.0f} spent"),
    (month_end, p50, f"median \\${p50:,.0f}"),
    (month_end, p95, f"95th pct \\${p95:,.0f}"),
    (month_end, BUDGET, f"budget \\${BUDGET:,.0f}"),
]
for x, y, text in end_labels:
    below = x == today  # keeps today's label clear of the budget pace and forecast lines
    ax.annotate(
        text, (x, y), xytext=(6, -8) if below else (6, 0), textcoords="offset points",
        va="top" if below else "center", color="#52514e",
    )
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%a %d %b"))

ax.set(title=f"{month:%B %Y}: cumulative spend vs budget ({headline_model} forecast)", ylabel="USD", xlabel="")
ax.set_xlim(month, month_end + timedelta(days=6))
ax.set_ylim(0, max(BUDGET, p95) * 1.08)
ax.legend(loc="upper left", frameon=False)
sns.despine()

In [ ]:
rolling = workday_rolling_mean(paced.filter(pl.col("day") <= today))
shown = paced.filter(pl.col("day") <= today)

fig, ax = plt.subplots(figsize=(12, 5))
for day in shown.filter(~pl.col("workday"))["day"]:
    start = datetime(day.year, day.month, day.day)
    ax.axvspan(start - timedelta(hours=12), start + timedelta(hours=12), color="#e6e5e0", lw=0, zorder=0)
for row in summary.iter_rows(named=True):
    m = row["month"]
    ax.hlines(
        row["daily_budget_usd"], m, month_days(m.year, m.month)[-1] + timedelta(days=1),
        color="#8a8983", lw=2, ls="--",
    )
ax.plot(shown["day"], shown["cost_usd"], color="#2a78d6", lw=1.5, alpha=0.55, label="Spent per day")
ax.plot(rolling["day"], rolling["rolling_usd"], color="#2a78d6", lw=2, label="Average, last 5 working days")
ax.plot([], [], color="#8a8983", lw=2, ls="--", label="Daily budget (per working day)")
ax.fill_between([], [], color="#e6e5e0", label="Weekend / holiday")

ax.set(title="Daily spend vs daily budget", ylabel="USD", xlabel="")
ax.set_xlim(first, today + timedelta(days=1))
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncols=4, frameon=False)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
sns.despine()

In [ ]:
progress = month_progress(paced.filter(pl.col("day") <= today), BUDGET)
months = progress["month"].unique().sort().to_list()
# Sequential blue: older months lighter, the current month darkest.
RAMP = ["#86b6ef", "#6da7ec", "#5598e7", "#3987e5", "#2a78d6", "#256abf", "#1c5cab", "#184f95", "#104281"]
ramp = [RAMP[round(i * (len(RAMP) - 1) / max(len(months) - 1, 1))] for i in range(len(months))]
if len(months) == 1:
    ramp = [RAMP[-1]]

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot([0, 100], [0, 100], color="#8a8983", lw=2, ls="--", label="Even pace")
ax.axhline(100, color="#8a8983", lw=1)
for m, color in zip(months, ramp):
    rows = progress.filter(pl.col("month") == m)
    ax.plot(rows["elapsed_pct"], rows["budget_used_pct"], color=color, lw=3 if m == month else 2, label=f"{m:%B %Y}")
    ax.annotate(
        f"{m:%b} {rows['budget_used_pct'][-1]:.0f}%", (rows["elapsed_pct"][-1], rows["budget_used_pct"][-1]),
        xytext=(6, 0), textcoords="offset points", va="center", color="#52514e",
    )
ax.set(
    title="Months compared: budget used vs working days elapsed",
    xlabel="Working days elapsed (%)", ylabel="Budget used (%)", xlim=(0, 108),
)
ax.set_ylim(0, max(110, progress["budget_used_pct"].max() * 1.08))
ax.legend(loc="upper left", frameon=False)
sns.despine()

In [ ]:
# Per month. The current month is up to today.
summary

## Forecast models

Three models simulate the rest of the month's daily spend, path by path; the bands are quantiles
of the running total across paths, so they widen the further out they reach.

- **Bootstrap**: resamples recent working days (in short runs) and off days. No trend.
- **ETS**: damped-trend exponential smoothing on log spend over working days; the trend bends
  instead of running on in a straight line.
- **Bayesian**: hurdle-lognormal spend with a damped local linear trend, fitted with NUTS.

In [ ]:
models: dict[ModelName, Forecaster] = {
    "bootstrap": BootstrapForecaster(),
    "ets": EtsForecaster(),
    "bayes": BayesForecaster(),
}
fig, axes = plt.subplots(1, len(models), figsize=(15, 4.5), sharey=True)
for (name, model), ax, color in zip(models.items(), axes, ("#2a78d6", "#eb6834", "#1baf7a")):
    ax.plot(to_date["day"], to_date["cum_spend"], color="#52514e", lw=2)
    ax.axhline(BUDGET, color="#8a8983", lw=1, ls="--")
    try:
        f = fan(spent_before, model(history, rest, N_PATHS, np.random.default_rng(SEED)), rest)
    except InsufficientHistory as e:
        ax.set_title(f"{name}: {e}", fontsize=9)
        continue
    draw_fan(ax, f, cutoff, spent_before, color)
    lo, mid, hi = np.quantile(f.month_end, [0.05, 0.5, 0.95])
    ax.set_title(
        f"{name}: median \\${mid:,.0f} (90% \\${lo:,.0f}–\\${hi:,.0f}), "
        f"P(over) {prob_over(f.month_end, BUDGET):.0%}",
        fontsize=10,
    )
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
    ax.set_xlim(month, month_end)
axes[0].set_ylabel("USD")
axes[0].legend(loc="upper left", frameon=False, fontsize=8)
sns.despine()

### Backtest

Each completed month is forecast from its 5th, 10th and 15th working day, seeing only the days
up to then, and scored against what was actually spent. **CRPS** (USD, lower is better) rewards
forecasts that are both right and sharp. **coverage90** should be near 90%: much lower means the
bands are too narrow, much higher that they are too wide. `linear` is the old point forecast.
With few months of history the coverage figure is noisy.

In [ ]:
backtest_models: dict[ModelName, Forecaster] = {
    **models,
    # Fewer draws per fit: the backtest refits at every origin.
    "bayes": BayesForecaster(num_warmup=300, num_samples=300),
    "linear": LinearForecaster(),
}
scores = backtest(
    history, backtest_models, backtest_origins(history), N_PATHS, np.random.default_rng(SEED)
)
backtest_summary(scores)

In [ ]:
# Every forecast in the backtest.
scores.sort("origin", "model")

## Spend per project

In [ ]:
def share(x: float) -> str:
    return "<1%" if x < 0.005 else f"{x:.0%}"


def pie(ax: plt.Axes, spend: pl.DataFrame, title: str) -> None:
    if spend.height == 0:
        ax.set_axis_off()
        ax.set_title(f"{title}\nno spend")
        return
    sliced = group_small(spend)
    total = sliced["cost_usd"].sum()
    names = sliced["project"].to_list()
    values = sliced["cost_usd"].to_list()
    wedges, _, _ = ax.pie(
        values,
        colors=[colors[n] for n in names],
        startangle=90,
        counterclock=False,
        autopct=lambda pct: f"{pct:.0f}%" if pct >= 5 else "",
        pctdistance=0.78,
        wedgeprops={"width": 0.45, "edgecolor": "white", "linewidth": 2},
        textprops={"color": "white", "fontweight": "bold"},
    )
    ax.set_title(f"{title}\n${total:,.2f}")
    ax.legend(
        wedges,
        [f"{n}  ${v:,.2f} ({share(v / total)})" for n, v in zip(names, values)],
        loc="upper center", bbox_to_anchor=(0.5, 0), frameon=False,
    )

fig, (left, right) = plt.subplots(1, 2, figsize=(12, 6.5))
pie(left, this_month, f"{since:%B %Y}")
pie(right, all_time, "All time")
fig.tight_layout()

In [ ]:
# The numbers behind the pies.
all_time.join(this_month, on="project", how="left", suffix="_this_month").fill_null(0.0)

## Daily spend

In [ ]:
days = daily_spend(hourly).to_pandas()
order = all_time["project"].to_list()
(
    so.Plot(days, x="day", y="cost_usd", color="project")
    .add(so.Bar(edgecolor="white", edgewidth=1, alpha=1), so.Stack())
    .scale(color=so.Nominal([colors[p] for p in order], order=order))
    .label(x="", y="USD", color="Project", title="Spend per day (local time)")
    .layout(size=(12, 4.5))
    .theme(plt.rcParams)
)

## Spend per model

In [ ]:
models = spend_by_model(sessions)
fig, ax = plt.subplots(figsize=(8, 0.6 * models.height + 1))
sns.barplot(models.to_pandas(), x="cost_usd", y="model", color="#2a78d6", saturation=1, ax=ax)
ax.bar_label(ax.containers[0], fmt="$%.2f", padding=4)
ax.set(xlabel="USD", ylabel="", title="Spend per model")
sns.despine(left=True)

## Most expensive sessions

In [ ]:
sessions.sort("cost_usd", descending=True).head(10).select(
    "project", "first_seen", "model", "branch", "runs", "cost_usd", "input_tokens", "output_tokens",
)